# Standalone Solar PV System Sizing

This walkthrough converts a 24-hour load profile into preliminary PV array, battery and inverter sizing metrics. It is an engineering screening tool, not a final design.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
ROOT=Path.cwd()
if not (ROOT/"src").exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/"src"))
from standalone_pv_design.sizing import design_system

cfg=json.loads((ROOT/"config/design.json").read_text())
load=pd.read_csv(ROOT/"data/demo/load_profile_synthetic.csv")["load_w"].to_numpy()
display(pd.DataFrame({"hour":range(24),"load_w":load}))

## 1. Daily load and peak demand

In [ ]:
daily_energy_kwh=load.sum()/1000
peak_load_w=load.max()
print("Daily energy (kWh):",round(daily_energy_kwh,3))
print("Peak hourly load (W):",peak_load_w)

plt.figure(figsize=(9,3))
plt.plot(range(24),load,marker="o")
plt.xlabel("Hour")
plt.ylabel("Load (W)")
plt.title("Synthetic daily load profile")
plt.grid(alpha=.2)
plt.show()

## 2. Preliminary system sizing

In [ ]:
design=design_system(
    load, cfg["peak_sun_hours"], cfg["autonomy_days"],
    cfg["battery_voltage_v"], cfg["system_derate"],
    cfg["depth_of_discharge"], cfg["battery_efficiency"],
    cfg["power_factor"], cfg["inverter_margin"]
)
pd.Series(design,name="design_value").to_frame()

## Engineering caveats

PV array sizing depends on worst-month solar resource, temperature, shading, tilt, orientation and system losses. Battery sizing depends on chemistry, temperature, ageing, allowable depth of discharge and reliability target. Inverter selection requires motor-starting and surge loads, not just hourly energy. Cable and protection studies require certified equipment data, fault current and applicable electrical standards.

Replace all synthetic inputs with site-specific values before design approval.